# 01 — Benchmark Final Model Penerjemah Duri → Indonesia (V5 CLEAN — 5 Model)

Notebook ini adalah **versi bersih** untuk mengulang benchmark pemilihan backbone LLM dari awal.

Benchmark menggunakan **53 data validation final** pada kondisi Zero-Shot yang sama. **Data test tidak digunakan untuk pemilihan backbone**, dan referensi Indonesia hanya digunakan setelah generasi untuk menghitung BLEU dan chrF++.

### Kandidat final kelas ±7–9B

| Kode | Model | Ukuran |
|---|---|---:|
| M1 | Llama 3.1 8B Instruct | 8B |
| M2 | Qwen2.5 7B Instruct | 7B |
| M3 | Gemma 2 9B IT | 9B |
| M4 | Mistral 7B Instruct v0.3 | 7B |
| M5 | Aya Expanse 8B | 8B |

### Konfigurasi benchmark

- dataset final: **417 train / 53 validation / 53 test = 523 pasangan**;
- benchmark hanya melakukan generasi pada **53 validation**;
- BF16 tanpa kuantisasi;
- greedy decoding (`do_sample=False`);
- `max_new_tokens=128`;
- satu keluaran per source;
- **chrF++ sebagai metrik utama** dan BLEU sebagai metrik pendukung;
- S3E dan MBR **tidak digunakan** pada benchmark pemilihan backbone;
- feasibility gate mencakup akses model, load BF16, inferensi, chat template, keluaran tidak kosong, dan ketersediaan target LoRA `q_proj` serta `v_proj`;
- backbone final dipilih dari model yang lolos feasibility gate berdasarkan **chrF++ tertinggi**, dengan **BLEU sebagai tie-breaker**.

### Penting untuk run bersih

Notebook ini **tidak membaca hasil benchmark lama**. Setiap run membuat folder baru dengan `RUN_ID` berbasis timestamp UTC. Checkpoint hanya berlaku di dalam run baru tersebut sehingga hasil benchmark sebelumnya tidak akan menyebabkan model dilewati.

Notebook ditujukan untuk **VS Code Remote SSH → Vast.ai RTX 4090 24 GB** dan tidak menginstal ulang PyTorch/CUDA bawaan image.


## 1. Instalasi dependensi

Tidak menginstal ulang PyTorch agar build CUDA bawaan Vast.ai tidak berubah.


In [1]:
# ============================================================
# 1. INSTALASI DEPENDENSI
# ============================================================
# PyTorch/CUDA sengaja TIDAK diinstal ulang agar build bawaan Vast.ai
# tetap digunakan.
%pip -q install -U \
    "transformers==4.48.1" \
    "accelerate==1.2.1" \
    "huggingface_hub==0.27.1" \
    "sacrebleu==2.5.1" \
    "sentencepiece==0.2.0" \
    "protobuf>=4.25,<6" \
    "pandas==2.2.3" \
    "numpy==1.26.4" \
    "psutil>=7,<8" \
    "openpyxl>=3.1,<4" \
    "tqdm>=4.66,<5"

print(
    "Dependensi selesai. Jika ini pertama kali dijalankan pada kernel baru, "
    "restart kernel SATU KALI, lalu mulai lagi dari Cell 2 (Import dan konfigurasi). "
    "Cell instalasi tidak perlu diulang setelah restart."
)


Note: you may need to restart the kernel to use updated packages.
Dependensi selesai. Jika ini pertama kali dijalankan pada kernel baru, restart kernel SATU KALI, lalu mulai lagi dari Cell 2 (Import dan konfigurasi). Cell instalasi tidak perlu diulang setelah restart.


## 2. Import dan konfigurasi

Lokasi proyek default:

```bash
/workspace/10122233_CikitaMeydilvira_Skripsi
```

Jika berbeda, set sebelum menjalankan notebook:

```bash
export SKRIPSI_PROJECT_ROOT=/workspace/10122233_CikitaMeydilvira_Skripsi
```


In [2]:
# ============================================================
# 2. IMPORT DAN KONFIGURASI
# ============================================================
from __future__ import annotations

import gc
import os
import re
import json
import time
import random
import hashlib
import shutil
import platform
import subprocess
import importlib.metadata as importlib_metadata
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import psutil
import torch

from packaging.version import Version
from tqdm import tqdm  # sengaja BUKAN tqdm.auto agar tidak memicu ipywidgets renderer
from IPython.display import display
from huggingface_hub import HfApi, get_token
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from sacrebleu.metrics import BLEU, CHRF

# ------------------------------------------------------------
# Environment penelitian
# ------------------------------------------------------------
TRANSFORMERS_EXPECTED = "4.48.1"
TRANSFORMERS_ACTUAL = importlib_metadata.version("transformers")

if Version(TRANSFORMERS_ACTUAL) != Version(TRANSFORMERS_EXPECTED):
    raise RuntimeError(
        "Versi Transformers tidak sesuai. "
        f"Ditemukan {TRANSFORMERS_ACTUAL}; diharapkan {TRANSFORMERS_EXPECTED}. "
        "Jalankan Cell 1, restart kernel bila diminta, lalu mulai dari Cell 2."
    )

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Lokasi proyek
# ------------------------------------------------------------
PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser()

# Dataset final yang sekarang digunakan berada langsung di folder Dataset.
DATASET_DIR = PROJECT_ROOT / "Dataset"
TRAIN_PATH = DATASET_DIR / "train.csv"
VALID_PATH = DATASET_DIR / "validation.csv"
TEST_PATH = DATASET_DIR / "test.csv"

OUTPUT_ROOT = PROJECT_ROOT / "Hasil" / "benchmark_model_translator"
CONFIG_DIR = PROJECT_ROOT / "Config"

# ------------------------------------------------------------
# Kandidat final ±7–9B
# ------------------------------------------------------------
MODELS: Dict[str, Dict[str, str]] = {
    "M1": {
        "display_name": "Llama 3.1 8B Instruct",
        "model_id": "meta-llama/Llama-3.1-8B-Instruct",
        "developer": "Meta",
        "size": "8B",
        "role": "Kandidat",
    },
    "M2": {
        "display_name": "Qwen2.5 7B Instruct",
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "developer": "Qwen / Alibaba",
        "size": "7B",
        "role": "Kandidat",
    },
    "M3": {
        "display_name": "Gemma 2 9B IT",
        "model_id": "google/gemma-2-9b-it",
        "developer": "Google",
        "size": "9B",
        "role": "Kandidat",
    },
    "M4": {
        "display_name": "Mistral 7B Instruct v0.3",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "developer": "Mistral AI",
        "size": "7B",
        "role": "Kandidat",
    },
    "M5": {
        "display_name": "Aya Expanse 8B",
        "model_id": "CohereLabs/aya-expanse-8b",
        "developer": "Cohere Labs",
        "size": "8B",
        "role": "Kandidat",
    },
}

EXPECTED_TRAIN_ROWS = 417
EXPECTED_VALID_ROWS = 53
EXPECTED_TEST_ROWS = 53
EXPECTED_TOTAL_ROWS = 523

MAX_NEW_TOKENS = 128
DO_SAMPLE = False
TEMPERATURE_RECORDED = 0.0
DTYPE = torch.bfloat16
DTYPE_NAME = "bfloat16"
DEVICE_INDEX = 0

LORA_TARGETS = ("q_proj", "v_proj")

RUN_HF_PREFLIGHT = True
RUN_SMOKE_TEST = True
RUN_FULL_BENCHMARK = True

# Exact-pair overlap dan conversation overlap adalah hard error.
# Exact-source overlap dengan reference berbeda hanya diagnostic warning.
STRICT_EXACT_SOURCE_OVERLAP = False

# Cache/token yang sudah dipakai pada Vast.ai saat ini.
DEFAULT_HF_HOME = (
    "/workspace/.hf_home"
    if Path("/workspace").exists()
    else str(Path.home() / ".cache" / "huggingface")
)
os.environ.setdefault("HF_HOME", DEFAULT_HF_HOME)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATASET_DIR  :", DATASET_DIR)
print("HF_HOME      :", os.environ["HF_HOME"])
print("Transformers :", TRANSFORMERS_ACTUAL)
print("Precision    :", "BF16 tanpa kuantisasi")
print("Decoding     :", "greedy / do_sample=False")
print("Kandidat     :", ", ".join(f"{k}={v['display_name']}" for k, v in MODELS.items()))


PROJECT_ROOT : /workspace/10122233_CikitaMeydilvira_Skripsi
DATASET_DIR  : /workspace/10122233_CikitaMeydilvira_Skripsi/Dataset
HF_HOME      : /workspace/.hf_home
Transformers : 4.48.1
Precision    : BF16 tanpa kuantisasi
Decoding     : greedy / do_sample=False
Kandidat     : M1=Llama 3.1 8B Instruct, M2=Qwen2.5 7B Instruct, M3=Gemma 2 9B IT, M4=Mistral 7B Instruct v0.3, M5=Aya Expanse 8B


## 3. Memuat split dataset final

Notebook hanya menerima:

- `Dataset/train.csv`
- `Dataset/validation.csv`
- `Dataset/test.csv`

Tidak ada fallback ke versi dataset lama.


In [3]:
# ============================================================
# 3. LOAD SPLIT DATASET FINAL
# ============================================================
for required_path in [TRAIN_PATH, VALID_PATH, TEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"File dataset final tidak ditemukan: {required_path}"
        )

def read_split(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )
    frame.columns = [str(c).strip() for c in frame.columns]
    return frame

df_train = read_split(TRAIN_PATH)
df_valid = read_split(VALID_PATH)
df_test = read_split(TEST_PATH)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_PATH :", TRAIN_PATH, "|", len(df_train), "baris")
print("VALID_PATH :", VALID_PATH, "|", len(df_valid), "baris")
print("TEST_PATH  :", TEST_PATH, "|", len(df_test), "baris")
print("TOTAL      :", len(df_train) + len(df_valid) + len(df_test))
display(df_valid.head())


TRAIN_PATH : /workspace/10122233_CikitaMeydilvira_Skripsi/Dataset/train.csv | 417 baris
VALID_PATH : /workspace/10122233_CikitaMeydilvira_Skripsi/Dataset/validation.csv | 53 baris
TEST_PATH  : /workspace/10122233_CikitaMeydilvira_Skripsi/Dataset/test.csv | 53 baris
TOTAL      : 523


,row_id,conversation_no,turn_order,duri_text,indonesia_text
0,C04_T01_A1,4,1,Indara sangantaq?,Siapa nama Anda?
1,C04_T02_B1,4,2,Disanganaq Nur.,Nama saya Nur.
2,C04_T03_A2,4,3,Melohnaq melajah bicara Duri.,Saya ingin belajar bahasa Duri.
3,C04_T04_B2,4,4,"Ieq, ke madoangkiq melajah.","Silahkan, kalau Anda ingin belajar."
4,C04_T05_A3,4,5,Apara bicara Durinna teq?,Apa bahasa Durinya ini?


## 4. Audit split dataset

Audit memblokir benchmark jika jumlah data, kolom wajib, `row_id`, exact-pair overlap, atau pemisahan conversation tidak sesuai. Exact-source overlap dengan reference berbeda hanya dicatat sebagai diagnostic warning.


In [4]:
# ============================================================
# 4. AUDIT SPLIT DATASET FINAL
# ============================================================
REQUIRED_COLUMNS = [
    "row_id",
    "conversation_no",
    "turn_order",
    "duri_text",
    "indonesia_text",
]

def normalize_text(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value).strip()).casefold()

def validate_split(
    frame: pd.DataFrame,
    split_name: str,
    expected_rows: int,
) -> pd.DataFrame:
    missing = [c for c in REQUIRED_COLUMNS if c not in frame.columns]
    if missing:
        raise ValueError(f"{split_name}: kolom wajib tidak ditemukan: {missing}")

    if len(frame) != expected_rows:
        raise ValueError(
            f"{split_name}: jumlah data harus {expected_rows}, ditemukan {len(frame)}."
        )

    clean = frame[REQUIRED_COLUMNS].copy().reset_index(drop=True)

    for col in REQUIRED_COLUMNS:
        empty_mask = clean[col].astype(str).str.strip().eq("")
        if empty_mask.any():
            bad = clean.loc[empty_mask, "row_id"].tolist()
            raise ValueError(f"{split_name}: nilai kosong pada kolom {col}: {bad}")

    if clean["row_id"].duplicated().any():
        bad = clean.loc[
            clean["row_id"].duplicated(keep=False), "row_id"
        ].tolist()
        raise ValueError(f"{split_name}: row_id duplikat: {bad}")

    clean["duri_norm"] = clean["duri_text"].map(normalize_text)
    clean["indonesia_norm"] = clean["indonesia_text"].map(normalize_text)
    clean["pair_norm"] = clean["duri_norm"] + " ||| " + clean["indonesia_norm"]

    if clean["pair_norm"].duplicated().any():
        bad = clean.loc[
            clean["pair_norm"].duplicated(keep=False),
            ["row_id", "duri_text", "indonesia_text"],
        ]
        raise ValueError(
            f"{split_name}: exact-pair duplicate masih ada:\n"
            + bad.to_string(index=False)
        )
    return clean

df_train_audit = validate_split(df_train, "train", EXPECTED_TRAIN_ROWS)
df_valid_audit = validate_split(df_valid, "validation", EXPECTED_VALID_ROWS)
df_test_audit = validate_split(df_test, "test", EXPECTED_TEST_ROWS)

if len(df_train_audit) + len(df_valid_audit) + len(df_test_audit) != EXPECTED_TOTAL_ROWS:
    raise ValueError("Total dataset final bukan 523 baris.")

split_frames = {
    "train": df_train_audit,
    "validation": df_valid_audit,
    "test": df_test_audit,
}

conversation_overlap_records = []
exact_pair_overlap_records = []
source_overlap_records = []

for left_name, right_name in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test"),
]:
    left = split_frames[left_name]
    right = split_frames[right_name]

    for conv in sorted(
        set(left["conversation_no"].astype(str))
        & set(right["conversation_no"].astype(str))
    ):
        conversation_overlap_records.append({
            "split_left": left_name,
            "split_right": right_name,
            "conversation_no": conv,
        })

    for pair in sorted(set(left["pair_norm"]) & set(right["pair_norm"])):
        exact_pair_overlap_records.append({
            "split_left": left_name,
            "split_right": right_name,
            "pair_norm": pair,
        })

    for source_norm in sorted(set(left["duri_norm"]) & set(right["duri_norm"])):
        left_rows = left[left["duri_norm"].eq(source_norm)]
        right_rows = right[right["duri_norm"].eq(source_norm)]
        for _, lrow in left_rows.iterrows():
            for _, rrow in right_rows.iterrows():
                source_overlap_records.append({
                    "split_left": left_name,
                    "split_right": right_name,
                    "duri_norm": source_norm,
                    "left_row_id": lrow["row_id"],
                    "right_row_id": rrow["row_id"],
                    "left_duri_text": lrow["duri_text"],
                    "right_duri_text": rrow["duri_text"],
                    "left_reference": lrow["indonesia_text"],
                    "right_reference": rrow["indonesia_text"],
                })

df_conversation_overlap = pd.DataFrame(conversation_overlap_records)
df_exact_pair_overlap = pd.DataFrame(exact_pair_overlap_records)
df_source_overlap = pd.DataFrame(source_overlap_records)

if not df_conversation_overlap.empty:
    raise RuntimeError(
        "Conversation overlap ditemukan lintas split:\n"
        + df_conversation_overlap.to_string(index=False)
    )

if not df_exact_pair_overlap.empty:
    raise RuntimeError(
        "Exact-pair overlap ditemukan lintas split:\n"
        + df_exact_pair_overlap.to_string(index=False)
    )

if not df_source_overlap.empty:
    msg = (
        f"PERINGATAN: ditemukan {len(df_source_overlap)} exact-source overlap lintas split. "
        "Dicatat sebagai diagnostic karena benchmark Zero-Shot tidak membaca train saat generasi."
    )
    if STRICT_EXACT_SOURCE_OVERLAP:
        raise RuntimeError(msg + "\n" + df_source_overlap.to_string(index=False))
    print(msg)

df_train = df_train_audit[REQUIRED_COLUMNS].copy()
df_valid = df_valid_audit[REQUIRED_COLUMNS].copy()
df_test = df_test_audit[REQUIRED_COLUMNS].copy()

df_split_audit = pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "conversations": frame["conversation_no"].nunique(),
        "exact_pair_duplicates_within_split": int(frame["pair_norm"].duplicated().sum()),
    }
    for name, frame in split_frames.items()
])

SPLIT_AUDIT_PATH = OUTPUT_ROOT / "split_audit_latest.csv"
df_split_audit.to_csv(SPLIT_AUDIT_PATH, index=False, encoding="utf-8-sig")

print("\nAudit split utama: LOLOS")
display(df_split_audit)
print("Conversation overlap lintas split:", len(df_conversation_overlap))
print("Exact-pair overlap lintas split   :", len(df_exact_pair_overlap))
print("Exact-source overlap diagnostic   :", len(df_source_overlap))


PERINGATAN: ditemukan 1 exact-source overlap lintas split. Dicatat sebagai diagnostic karena benchmark Zero-Shot tidak membaca train saat generasi.

Audit split utama: LOLOS


,split,rows,conversations,exact_pair_duplicates_within_split
0,train,417,44,0
1,validation,53,3,0
2,test,53,3,0


Conversation overlap lintas split: 0
Exact-pair overlap lintas split   : 0
Exact-source overlap diagnostic   : 1


## 5. Prompt Zero-Shot

Prompt dibangun hanya dari `duri_text`. Reference Indonesia tidak diberikan saat generasi.


In [5]:
# ============================================================
# 5. PROMPT ZERO-SHOT
# ============================================================
PROMPT_TEMPLATE = """Terjemahkan teks bahasa Duri berikut ke bahasa Indonesia.
Keluarkan hanya hasil terjemahan bahasa Indonesia tanpa penjelasan tambahan.

Teks bahasa Duri:
{duri_text}"""

def build_translation_instruction(duri_text: str) -> str:
    return PROMPT_TEMPLATE.format(duri_text=str(duri_text).strip())

PROMPT_HASH = hashlib.sha256(
    PROMPT_TEMPLATE.encode("utf-8")
).hexdigest()[:16]

preview = build_translation_instruction(df_valid.iloc[0]["duri_text"])
print("Prompt hash:", PROMPT_HASH)
print("\nPreview:\n", preview)

# Reference tidak boleh muncul karena dimasukkan ke prompt.
# Cek dilakukan secara struktural: fungsi hanya menerima duri_text.
assert "indonesia_text" not in PROMPT_TEMPLATE


Prompt hash: d9b5730e7563b57f

Preview:
 Terjemahkan teks bahasa Duri berikut ke bahasa Indonesia.
Keluarkan hanya hasil terjemahan bahasa Indonesia tanpa penjelasan tambahan.

Teks bahasa Duri:
Indara sangantaq?


## 6. Pemeriksaan GPU dan BF16


In [6]:
# ============================================================
# 6. PEMERIKSAAN GPU DAN BF16
# ============================================================
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA tidak tersedia. Pastikan notebook berjalan pada kernel remote Vast.ai."
    )

torch.cuda.set_device(DEVICE_INDEX)

if not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU tidak mendukung BF16. Tidak ada fallback otomatis ke FP16.")

gpu_name = torch.cuda.get_device_name(DEVICE_INDEX)
gpu_props = torch.cuda.get_device_properties(DEVICE_INDEX)
gpu_total_gib = gpu_props.total_memory / (1024 ** 3)

print("GPU        :", gpu_name)
print("VRAM total :", f"{gpu_total_gib:.2f} GiB")
print("BF16       :", torch.cuda.is_bf16_supported())

if gpu_total_gib < 24:
    print(
        "PERINGATAN: VRAM < 24 GiB. Gemma 2 9B BF16 dapat mendekati "
        "batas VRAM tergantung overhead runtime."
    )

try:
    p = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free,driver_version",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    print("\nnvidia-smi:\n" + p.stdout.strip())
except Exception as exc:
    print("nvidia-smi gagal dibaca:", exc)

cache = Path(os.environ["HF_HOME"])
cache.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage(cache)
disk_free_gib = disk.free / (1024 ** 3)

print("\nDisk cache tersedia:", f"{disk_free_gib:.2f} GiB")

if disk_free_gib < 100:
    print(
        "PERINGATAN: ruang cache < 100 GiB. Lima checkpoint BF16 dapat "
        "membutuhkan ruang besar jika cache tidak dipertahankan."
    )


GPU        : NVIDIA GeForce RTX 4090
VRAM total : 23.52 GiB
BF16       : True
PERINGATAN: VRAM < 24 GiB. Gemma 2 9B BF16 dapat mendekati batas VRAM tergantung overhead runtime.

nvidia-smi:
NVIDIA GeForce RTX 4090, 24564 MiB, 23688 MiB, 595.58.03

Disk cache tersedia: 122.91 GiB


## 7. Preflight Hugging Face


In [7]:
# ============================================================
# 7. PREFLIGHT HUGGING FACE
# ============================================================
HF_TOKEN = get_token() or os.getenv("HF_TOKEN", "").strip() or None
hf_api = HfApi(token=HF_TOKEN)

preflight = []

if RUN_HF_PREFLIGHT:
    for key, info in MODELS.items():
        print(f"Memeriksa {key}: {info['model_id']}")
        try:
            m = hf_api.model_info(repo_id=info["model_id"], revision="main")
            rec = {
                "model_key": key,
                "display_name": info["display_name"],
                "model_id": info["model_id"],
                "access_ok": True,
                "revision_sha": str(m.sha or ""),
                "gated": str(m.gated),
                "private": bool(m.private),
                "error": "",
            }
            print("  OK | revision:", rec["revision_sha"][:12])
        except Exception as exc:
            rec = {
                "model_key": key,
                "display_name": info["display_name"],
                "model_id": info["model_id"],
                "access_ok": False,
                "revision_sha": "",
                "gated": "",
                "private": "",
                "error": f"{type(exc).__name__}: {exc}",
            }
            print("  GAGAL:", rec["error"])
        preflight.append(rec)

    df_preflight = pd.DataFrame(preflight)
    display(df_preflight)

    if not df_preflight["access_ok"].all():
        raise RuntimeError(
            "Ada model yang belum dapat diakses. Pastikan `hf auth whoami` berhasil "
            "dan license model gated sudah diterima."
        )
else:
    df_preflight = pd.DataFrame()
    print("Preflight dilewati.")

MODEL_REVISIONS = {
    r["model_key"]: r["revision_sha"]
    for r in preflight
    if r["access_ok"]
}


Memeriksa M1: meta-llama/Llama-3.1-8B-Instruct
  OK | revision: 0e9e39f249a1
Memeriksa M2: Qwen/Qwen2.5-7B-Instruct
  OK | revision: a09a35458c70
Memeriksa M3: google/gemma-2-9b-it
  OK | revision: 11c9b309abf7
Memeriksa M4: mistralai/Mistral-7B-Instruct-v0.3
  OK | revision: c170c708c41d
Memeriksa M5: CohereLabs/aya-expanse-8b
  OK | revision: 5062468bf9bc


,model_key,display_name,model_id,access_ok,revision_sha,gated,private,error
0,M1,Llama 3.1 8B Instruct,meta-llama/Llama-3.1-8B-Instruct,True,0e9e39f249a16976918f6564b8830bc894c89659,manual,False,
1,M2,Qwen2.5 7B Instruct,Qwen/Qwen2.5-7B-Instruct,True,a09a35458c702b33eeacc393d103063234e8bc28,False,False,
2,M3,Gemma 2 9B IT,google/gemma-2-9b-it,True,11c9b309abf73637e4b6f9a3fa1e92e615547819,manual,False,
3,M4,Mistral 7B Instruct v0.3,mistralai/Mistral-7B-Instruct-v0.3,True,c170c708c41dac9275d15a8fff4eca08d52bab71,False,False,
4,M5,Aya Expanse 8B,CohereLabs/aya-expanse-8b,True,5062468bf9bc0c6035fd64e06274333ec127d980,auto,False,


## 8. Run ID dan folder output baru

Setiap eksekusi membuat folder **baru**. Notebook tidak akan memakai checkpoint benchmark lama.

Jika benchmark terputus di tengah **tanpa restart kernel**, jalankan ulang Cell 12 dan checkpoint dalam `RUN_DIR` yang sama akan dipakai. Jika kernel direstart, jalankan notebook dari Cell 2 dan sebuah run baru akan dibuat.


In [8]:
# ============================================================
# 8. RUN ID DAN KONFIGURASI — SELALU FRESH RUN
# ============================================================
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not-installed"

DATASET_HASH = sha256_file(VALID_PATH)[:16]

# Timestamp + hash pendek: selalu membuat folder baru untuk run baru.
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(
        {
            "dataset_hash": DATASET_HASH,
            "prompt_hash": PROMPT_HASH,
            "models": MODELS,
            "seed": SEED,
            "dtype": DTYPE_NAME,
            "max_new_tokens": MAX_NEW_TOKENS,
        },
        sort_keys=True,
        ensure_ascii=False,
    ).encode("utf-8")
).hexdigest()[:8]

RUN_ID = f"{RUN_STAMP}_{RUN_FINGERPRINT}"
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

CONFIG_PATH = RUN_DIR / "run_config.json"
PROMPT_PATH = RUN_DIR / "prompt_zero_shot.txt"
RUNTIME_PATH = RUN_DIR / "model_runtime.csv"
SUMMARY_PATH = RUN_DIR / "benchmark_summary.csv"
ALL_PREDICTIONS_PATH = RUN_DIR / "predictions_all.csv"
REPORT_XLSX_PATH = RUN_DIR / "benchmark_model_report.xlsx"
PREFLIGHT_PATH = RUN_DIR / "hf_preflight.csv"
SMOKE_PATH = RUN_DIR / "smoke_test.csv"

run_config = {
    "created_at_utc": utc_now_iso(),
    "run_id": RUN_ID,
    "fresh_run": True,
    "project_root": str(PROJECT_ROOT),
    "train_path": str(TRAIN_PATH),
    "valid_path": str(VALID_PATH),
    "test_path": str(TEST_PATH),
    "dataset_hash_validation": DATASET_HASH,
    "prompt_hash": PROMPT_HASH,
    "models": MODELS,
    "model_revisions": MODEL_REVISIONS,
    "expected_rows": {
        "train": EXPECTED_TRAIN_ROWS,
        "validation": EXPECTED_VALID_ROWS,
        "test": EXPECTED_TEST_ROWS,
        "total": EXPECTED_TOTAL_ROWS,
    },
    "precision": DTYPE_NAME,
    "quantization": None,
    "generation": {
        "do_sample": DO_SAMPLE,
        "temperature_recorded": TEMPERATURE_RECORDED,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
    "lora_targets_checked": list(LORA_TARGETS),
    "seed": SEED,
    "gpu_name": gpu_name,
    "gpu_total_gib": gpu_total_gib,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "package_versions": {
        "transformers": package_version("transformers"),
        "accelerate": package_version("accelerate"),
        "huggingface_hub": package_version("huggingface_hub"),
        "sacrebleu": package_version("sacrebleu"),
        "pandas": package_version("pandas"),
        "numpy": package_version("numpy"),
        "psutil": package_version("psutil"),
    },
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

with open(PROMPT_PATH, "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

if not df_preflight.empty:
    df_preflight.to_csv(PREFLIGHT_PATH, index=False, encoding="utf-8-sig")

print("FRESH RUN")
print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)
print("Tidak membaca checkpoint benchmark lama.")


FRESH RUN
RUN_ID : 20260813T051727Z_88f26a8d
RUN_DIR: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_model_translator/runs/20260813T051727Z_88f26a8d
Tidak membaca checkpoint benchmark lama.


## 9. Pascaproses teknis


In [9]:
# ============================================================
# 9. PASCAPROSES TEKNIS
# ============================================================
OUTPUT_PREFIXES = [
    "Bahasa Indonesia:",
    "Indonesia:",
    "Terjemahan:",
    "Hasil terjemahan:",
]

def clean_translation(text: str) -> str:
    x = str(text or "").strip()
    for _ in range(3):
        before = x
        for prefix in OUTPUT_PREFIXES:
            if x.lower().startswith(prefix.lower()):
                x = x[len(prefix):].lstrip()
                break
        if x == before:
            break
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\s*\n\s*", " ", x)
    x = re.sub(r"\s{2,}", " ", x)
    return x.strip()

assert clean_translation("Bahasa Indonesia:  Saya pergi. ") == "Saya pergi."
assert clean_translation(" Halo   dunia\n") == "Halo dunia"
print("Self-test pascaproses: OK")


Self-test pascaproses: OK


## 10. Helper pemuatan model dan generasi


In [10]:
# ============================================================
# 10. HELPER MODEL DAN GENERASI
# ============================================================
def gpu_sync() -> None:
    if torch.cuda.is_available():
        torch.cuda.synchronize(DEVICE_INDEX)

def clear_gpu_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def unload_model(
    model: Optional[torch.nn.Module],
    tokenizer: Optional[Any],
) -> None:
    # Referensi lokal akan hilang ketika fungsi pemanggil meng-overwrite
    # variabel model/tokenizer. Di sini kita fokus membersihkan cache CUDA.
    try:
        if model is not None:
            model.to("cpu")
    except Exception:
        pass
    del model
    del tokenizer
    clear_gpu_memory()

def inspect_lora_targets(
    model: torch.nn.Module,
) -> Tuple[bool, Dict[str, int]]:
    names = [name for name, _ in model.named_modules()]
    counts = {
        target: sum(name.endswith(target) for name in names)
        for target in LORA_TARGETS
    }
    return all(value > 0 for value in counts.values()), counts

def load_tokenizer(model_id: str, revision: str):
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            revision=revision,
            token=HF_TOKEN,
            use_fast=True,
            trust_remote_code=False,
        )
        tokenizer_backend = "fast"
    except Exception as first_exc:
        print(
            f"Fast tokenizer gagal untuk {model_id}; mencoba slow tokenizer. "
            f"Error awal: {first_exc}"
        )
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            revision=revision,
            token=HF_TOKEN,
            use_fast=False,
            trust_remote_code=False,
        )
        tokenizer_backend = "slow"
    return tokenizer, tokenizer_backend

def load_model(
    model_key: str,
    model_info: Dict[str, str],
) -> Tuple[Any, torch.nn.Module, Dict[str, Any]]:
    model_id = model_info["model_id"]
    revision = MODEL_REVISIONS.get(model_key, "main")

    clear_gpu_memory()
    start = time.perf_counter()

    tokenizer, tokenizer_backend = load_tokenizer(model_id, revision)

    if not getattr(tokenizer, "chat_template", None):
        raise RuntimeError(f"{model_id} tidak memiliki chat_template.")

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        token=HF_TOKEN,
        torch_dtype=DTYPE,
        device_map={"": DEVICE_INDEX},
        low_cpu_mem_usage=True,
        trust_remote_code=False,
    )

    model.eval()
    model.config.use_cache = True

    if bool(getattr(model, "is_loaded_in_4bit", False)) or bool(
        getattr(model, "is_loaded_in_8bit", False)
    ):
        raise RuntimeError(f"{model_id} terdeteksi dimuat dalam 4-bit/8-bit.")

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise RuntimeError(
                f"{model_id} tidak memiliki pad_token_id maupun eos_token_id."
            )
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Greedy benchmark. Null-kan sampling-only defaults agar warning
    # temperature/top_p tidak mengganggu output notebook.
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None
        model.generation_config.pad_token_id = tokenizer.pad_token_id

    first_param = next(model.parameters())
    if first_param.device.type != "cuda":
        raise RuntimeError(
            f"{model_id} tidak dimuat di GPU. Device: {first_param.device}"
        )

    dtype_numel: Dict[str, int] = {}
    for parameter in model.parameters():
        if parameter.is_floating_point():
            dtype_name = str(parameter.dtype)
            dtype_numel[dtype_name] = dtype_numel.get(dtype_name, 0) + int(parameter.numel())

    if not dtype_numel:
        raise RuntimeError(f"{model_id} tidak memiliki parameter floating-point.")

    total_float_numel = sum(dtype_numel.values())
    bf16_key = str(torch.bfloat16)
    bf16_numel = dtype_numel.get(bf16_key, 0)
    bf16_ratio = bf16_numel / total_float_numel if total_float_numel else 0.0
    dominant_dtype = max(dtype_numel, key=dtype_numel.get)

    print(f"{model_id} dtype distribution:", dtype_numel)
    print(f"{model_id} BF16 parameter ratio:", f"{bf16_ratio:.6%}")
    print(f"{model_id} dominant dtype:", dominant_dtype)

    if dominant_dtype != bf16_key:
        raise RuntimeError(
            f"{model_id} tidak didominasi parameter BF16. Distribusi: {dtype_numel}"
        )

    lora_ok, counts = inspect_lora_targets(model)
    gpu_sync()

    metadata = {
        "model_key": model_key,
        "display_name": model_info["display_name"],
        "model_id": model_id,
        "revision": revision,
        "load_success": True,
        "load_seconds": time.perf_counter() - start,
        "chat_template_available": True,
        "tokenizer_backend": tokenizer_backend,
        "lora_qv_compatible": lora_ok,
        "lora_target_counts": json.dumps(counts, ensure_ascii=False),
        "model_class": model.__class__.__name__,
        "tokenizer_class": tokenizer.__class__.__name__,
        "dtype_distribution_numel": json.dumps(
            dtype_numel, ensure_ascii=False, sort_keys=True
        ),
        "bf16_parameter_ratio": bf16_ratio,
        "dominant_dtype": dominant_dtype,
        "attn_logit_softcapping": getattr(
            model.config, "attn_logit_softcapping", None
        ),
        "final_logit_softcapping": getattr(
            model.config, "final_logit_softcapping", None
        ),
        "error_class": "",
        "error_message": "",
    }

    return tokenizer, model, metadata

def generate_translation(
    tokenizer: Any,
    model: torch.nn.Module,
    duri_text: str,
) -> Dict[str, Any]:
    messages = [{
        "role": "user",
        "content": build_translation_instruction(duri_text),
    }]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    encoded = {
        key: value.to(f"cuda:{DEVICE_INDEX}")
        for key, value in encoded.items()
    }

    prompt_len = int(encoded["input_ids"].shape[-1])

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if tokenizer.eos_token_id is not None:
        generation_kwargs["eos_token_id"] = tokenizer.eos_token_id

    gpu_sync()
    start = time.perf_counter()

    with torch.inference_mode():
        generated = model.generate(**encoded, **generation_kwargs)

    gpu_sync()
    latency = time.perf_counter() - start

    new_tokens = generated[0, prompt_len:]
    raw = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()
    clean = clean_translation(raw)

    return {
        "raw_output": raw,
        "clean_output": clean,
        "prompt_tokens": prompt_len,
        "completion_tokens": int(new_tokens.shape[-1]),
        "latency_seconds": latency,
        "nonempty_output": bool(clean.strip()),
        "hit_max_new_tokens": int(new_tokens.shape[-1]) >= MAX_NEW_TOKENS,
    }

print("Helper model/generation siap.")


Helper model/generation siap.


## 11. Feasibility smoke test satu kasus per model


In [11]:
# ============================================================
# 11. FEASIBILITY SMOKE TEST SATU KASUS PER MODEL
# ============================================================
smoke = []
SMOKE_SUCCESS: Dict[str, bool] = {}

if RUN_SMOKE_TEST:
    source = df_valid.iloc[0]["duri_text"]

    for key, info in MODELS.items():
        tokenizer = None
        model = None
        print(f"\nSmoke test {key} - {info['display_name']}")

        try:
            tokenizer, model, meta = load_model(key, info)
            result = generate_translation(tokenizer, model, source)

            if not result["nonempty_output"]:
                raise RuntimeError("Keluaran smoke test kosong.")

            if not bool(meta["lora_qv_compatible"]):
                raise RuntimeError(
                    f"Target LoRA {LORA_TARGETS} tidak ditemukan lengkap."
                )

            rec = {
                "model_key": key,
                "display_name": info["display_name"],
                "success": True,
                "output": result["clean_output"],
                "latency_seconds": result["latency_seconds"],
                "completion_tokens": result["completion_tokens"],
                "hit_max_new_tokens": result["hit_max_new_tokens"],
                "lora_qv_compatible": meta["lora_qv_compatible"],
                "error": "",
            }
            print("Berhasil:", result["clean_output"])
        except Exception as exc:
            rec = {
                "model_key": key,
                "display_name": info["display_name"],
                "success": False,
                "output": "",
                "latency_seconds": np.nan,
                "completion_tokens": np.nan,
                "hit_max_new_tokens": False,
                "lora_qv_compatible": False,
                "error": f"{type(exc).__name__}: {exc}",
            }
            print("GAGAL:", rec["error"])
        finally:
            unload_model(model, tokenizer)
            model = None
            tokenizer = None

        smoke.append(rec)
        SMOKE_SUCCESS[key] = bool(rec["success"])

    df_smoke = pd.DataFrame(smoke)
    df_smoke.to_csv(SMOKE_PATH, index=False, encoding="utf-8-sig")
    display(df_smoke)
    print(
        f"Smoke test selesai: {sum(SMOKE_SUCCESS.values())}/{len(MODELS)} "
        "kandidat berhasil inferensi."
    )
else:
    df_smoke = pd.DataFrame()
    SMOKE_SUCCESS = {k: True for k in MODELS}
    print("Smoke test dilewati.")



Smoke test M1 - Llama 3.1 8B Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

meta-llama/Llama-3.1-8B-Instruct dtype distribution: {'torch.bfloat16': 8030261248}
meta-llama/Llama-3.1-8B-Instruct BF16 parameter ratio: 100.000000%
meta-llama/Llama-3.1-8B-Instruct dominant dtype: torch.bfloat16
Berhasil: Siapa yang datang?

Smoke test M2 - Qwen2.5 7B Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen/Qwen2.5-7B-Instruct dtype distribution: {'torch.bfloat16': 7615616512}
Qwen/Qwen2.5-7B-Instruct BF16 parameter ratio: 100.000000%
Qwen/Qwen2.5-7B-Instruct dominant dtype: torch.bfloat16
Berhasil: Indara senang?

Smoke test M3 - Gemma 2 9B IT


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

... (4 outputs left out)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

CohereLabs/aya-expanse-8b dtype distribution: {'torch.bfloat16': 8028033024}
CohereLabs/aya-expanse-8b BF16 parameter ratio: 100.000000%
CohereLabs/aya-expanse-8b dominant dtype: torch.bfloat16
Berhasil: Apa indra?


,model_key,display_name,success,output,latency_seconds,completion_tokens,hit_max_new_tokens,lora_qv_compatible,error
0,M1,Llama 3.1 8B Instruct,True,Siapa yang datang?,0.746492,7,False,True,
1,M2,Qwen2.5 7B Instruct,True,Indara senang?,0.168094,6,False,True,
2,M3,Gemma 2 9B IT,True,Indara bagaimana?,0.393728,8,False,True,
3,M4,Mistral 7B Instruct v0.3,True,Indara siapa itu? (Artinya: Siapa Indara?),0.447848,18,False,True,
4,M5,Aya Expanse 8B,True,Apa indra?,0.162735,6,False,True,


Smoke test selesai: 5/5 kandidat berhasil inferensi.


### 11A. Sanity check khusus Gemma 2


In [12]:
# ============================================================
# 11A. SANITY CHECK KHUSUS GEMMA 2
# ============================================================
if RUN_SMOKE_TEST:
    gemma_mask = df_smoke["model_key"].eq("M3")
    gemma_row = df_smoke[gemma_mask]

    if gemma_row.empty:
        print("Sanity check Gemma 2 dilewati: hasil smoke M3 tidak ditemukan.")
    elif not bool(gemma_row.iloc[-1]["success"]):
        print("Sanity check Gemma 2 dilewati karena M3 sudah gagal smoke test.")
    else:
        gemma_output = str(gemma_row.iloc[-1]["output"]).strip()
        print("Smoke output M3 / Gemma 2:")
        print(gemma_output)

        tokens = gemma_output.split()
        repetitive = False

        if len(tokens) >= 8:
            unique_ratio = len(set(tokens)) / len(tokens)
            repetitive = unique_ratio < 0.20
            print("Unique-token ratio:", round(unique_ratio, 4))

        if repetitive:
            warning = (
                "Sanity check Gemma 2 mendeteksi repetisi ekstrem. "
                "M3 ditandai gagal feasibility dan tidak akan dijalankan "
                "pada benchmark penuh."
            )
            print("PERINGATAN:", warning)
            df_smoke.loc[gemma_mask, "success"] = False
            df_smoke.loc[gemma_mask, "error"] = warning
            SMOKE_SUCCESS["M3"] = False
            df_smoke.to_csv(SMOKE_PATH, index=False, encoding="utf-8-sig")
        else:
            print("Sanity check teknis Gemma 2: lolos.")


Smoke output M3 / Gemma 2:
Indara bagaimana?
Sanity check teknis Gemma 2: lolos.


## 12. Benchmark penuh dengan checkpoint

Checkpoint disimpan **hanya di folder RUN baru ini**. Tidak ada hasil benchmark lama yang dibaca.


In [13]:
# ============================================================
# 12. BENCHMARK PENUH DENGAN CHECKPOINT
# ============================================================
PREDICTION_COLUMNS = [
    "timestamp_utc",
    "run_id",
    "model_key",
    "display_name",
    "model_id",
    "revision",
    "row_id",
    "source_text",
    "reference_text",
    "raw_output",
    "clean_output",
    "generation_success",
    "nonempty_output",
    "hit_max_new_tokens",
    "prompt_tokens",
    "completion_tokens",
    "latency_seconds",
    "error_class",
    "error_message",
]

def checkpoint_path(key: str) -> Path:
    return RUN_DIR / f"predictions_{key}.csv"

def load_checkpoint(key: str) -> pd.DataFrame:
    p = checkpoint_path(key)
    if not p.exists():
        return pd.DataFrame(columns=PREDICTION_COLUMNS)
    f = pd.read_csv(
        p,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )
    return (
        f.drop_duplicates(subset=["row_id"], keep="last")
        .reset_index(drop=True)
    )

def save_checkpoint(key: str, frame: pd.DataFrame) -> None:
    (
        frame[PREDICTION_COLUMNS]
        .drop_duplicates(subset=["row_id"], keep="last")
        .reset_index(drop=True)
        .to_csv(
            checkpoint_path(key),
            index=False,
            encoding="utf-8-sig",
        )
    )

def successful_ids(frame: pd.DataFrame) -> set:
    if frame.empty:
        return set()
    mask = (
        frame["generation_success"]
        .astype(str)
        .str.lower()
        .eq("true")
    )
    return set(frame.loc[mask, "row_id"])

runtime_map: Dict[str, Dict[str, Any]] = {}

def save_runtime_map() -> pd.DataFrame:
    ordered = [runtime_map[k] for k in MODELS if k in runtime_map]
    frame = pd.DataFrame(ordered)
    frame.to_csv(RUNTIME_PATH, index=False, encoding="utf-8-sig")
    return frame

def empty_runtime(key: str, info: Dict[str, str]) -> Dict[str, Any]:
    return {
        "model_key": key,
        "display_name": info["display_name"],
        "model_id": info["model_id"],
        "revision": MODEL_REVISIONS.get(key, "main"),
        "load_success": False,
        "load_seconds": np.nan,
        "chat_template_available": False,
        "tokenizer_backend": "",
        "lora_qv_compatible": False,
        "lora_target_counts": "",
        "model_class": "",
        "tokenizer_class": "",
        "dtype_distribution_numel": "",
        "bf16_parameter_ratio": np.nan,
        "dominant_dtype": "",
        "attn_logit_softcapping": np.nan,
        "final_logit_softcapping": np.nan,
        "peak_allocated_gib": np.nan,
        "peak_reserved_gib": np.nan,
        "error_class": "",
        "error_message": "",
        "resume_from_checkpoint": False,
    }

def benchmark_one_model(
    key: str,
    info: Dict[str, str],
) -> Dict[str, Any]:
    checkpoint = load_checkpoint(key)
    done = successful_ids(checkpoint)

    if len(done) == EXPECTED_VALID_ROWS:
        print(f"{key} sudah lengkap di checkpoint RUN INI; dilewati.")
        recovered = runtime_map.get(key, empty_runtime(key, info))
        recovered["resume_from_checkpoint"] = True
        return recovered

    tokenizer = None
    model = None
    runtime = empty_runtime(key, info)

    try:
        print(f"\nMemuat {key} - {info['display_name']}...")
        tokenizer, model, meta = load_model(key, info)
        runtime.update(meta)
        runtime["load_success"] = True

        torch.cuda.reset_peak_memory_stats(DEVICE_INDEX)

        pending = df_valid[~df_valid["row_id"].isin(done)].copy()
        print("Data yang perlu dijalankan:", len(pending))

        checkpoint_map = {
            str(r["row_id"]): r.to_dict()
            for _, r in checkpoint.iterrows()
        }

        progress = tqdm(
            pending.iterrows(),
            total=len(pending),
            desc=f"{key} {info['display_name']}",
            unit="baris",
            dynamic_ncols=False,
            leave=True,
        )

        for _, row in progress:
            rec = {
                "timestamp_utc": utc_now_iso(),
                "run_id": RUN_ID,
                "model_key": key,
                "display_name": info["display_name"],
                "model_id": info["model_id"],
                "revision": MODEL_REVISIONS.get(key, "main"),
                "row_id": row["row_id"],
                "source_text": row["duri_text"],
                "reference_text": row["indonesia_text"],
                "raw_output": "",
                "clean_output": "",
                "generation_success": False,
                "nonempty_output": False,
                "hit_max_new_tokens": False,
                "prompt_tokens": "",
                "completion_tokens": "",
                "latency_seconds": "",
                "error_class": "",
                "error_message": "",
            }

            try:
                res = generate_translation(
                    tokenizer,
                    model,
                    row["duri_text"],
                )
                rec.update({
                    "raw_output": res["raw_output"],
                    "clean_output": res["clean_output"],
                    "generation_success": True,
                    "nonempty_output": res["nonempty_output"],
                    "hit_max_new_tokens": res["hit_max_new_tokens"],
                    "prompt_tokens": res["prompt_tokens"],
                    "completion_tokens": res["completion_tokens"],
                    "latency_seconds": res["latency_seconds"],
                })

                if not res["nonempty_output"]:
                    rec["generation_success"] = False
                    rec["error_class"] = "EmptyOutput"
                    rec["error_message"] = "Keluaran setelah pascaproses kosong."

            except torch.cuda.OutOfMemoryError as exc:
                rec.update({
                    "error_class": type(exc).__name__,
                    "error_message": str(exc)[:4000],
                })
                checkpoint_map[str(row["row_id"])] = rec
                save_checkpoint(key, pd.DataFrame(checkpoint_map.values()))
                raise

            except Exception as exc:
                rec.update({
                    "error_class": type(exc).__name__,
                    "error_message": str(exc)[:4000],
                })

            checkpoint_map[str(row["row_id"])] = rec
            save_checkpoint(key, pd.DataFrame(checkpoint_map.values()))

        gpu_sync()
        runtime["peak_allocated_gib"] = (
            torch.cuda.max_memory_allocated(DEVICE_INDEX) / (1024 ** 3)
        )
        runtime["peak_reserved_gib"] = (
            torch.cuda.max_memory_reserved(DEVICE_INDEX) / (1024 ** 3)
        )

        return runtime

    except Exception as exc:
        runtime["error_class"] = type(exc).__name__
        runtime["error_message"] = str(exc)[:4000]
        print("GAGAL:", runtime["error_class"], runtime["error_message"])
        return runtime

    finally:
        unload_model(model, tokenizer)
        model = None
        tokenizer = None

if not RUN_FULL_BENCHMARK:
    raise RuntimeError("RUN_FULL_BENCHMARK=False.")

for key, info in MODELS.items():
    if not bool(SMOKE_SUCCESS.get(key, False)):
        skipped = empty_runtime(key, info)
        skipped.update({
            "error_class": "SmokeTestFailed",
            "error_message": (
                df_smoke.loc[
                    df_smoke["model_key"].eq(key),
                    "error",
                ].iloc[-1]
                if not df_smoke.empty
                and df_smoke["model_key"].eq(key).any()
                else "Smoke test gagal."
            ),
        })
        runtime_map[key] = skipped
        save_runtime_map()
        print(f"{key} dilewati pada benchmark penuh karena smoke test gagal.")
        continue

    runtime_map[key] = benchmark_one_model(key, info)
    save_runtime_map()

df_runtime = save_runtime_map()
display(df_runtime)

print("\nBenchmark loop selesai.")
print("RUN_DIR:", RUN_DIR)



Memuat M1 - Llama 3.1 8B Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

meta-llama/Llama-3.1-8B-Instruct dtype distribution: {'torch.bfloat16': 8030261248}
meta-llama/Llama-3.1-8B-Instruct BF16 parameter ratio: 100.000000%
meta-llama/Llama-3.1-8B-Instruct dominant dtype: torch.bfloat16
Data yang perlu dijalankan: 53


M1 Llama 3.1 8B Instruct: 100%|██████████| 53/53 [00:24<00:00,  2.18baris/s]



Memuat M2 - Qwen2.5 7B Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

... (12 outputs left out)

CohereLabs/aya-expanse-8b dtype distribution: {'torch.bfloat16': 8028033024}
CohereLabs/aya-expanse-8b BF16 parameter ratio: 100.000000%
CohereLabs/aya-expanse-8b dominant dtype: torch.bfloat16
Data yang perlu dijalankan: 53


M5 Aya Expanse 8B: 100%|██████████| 53/53 [00:22<00:00,  2.35baris/s]


,model_key,display_name,model_id,revision,load_success,load_seconds,chat_template_available,tokenizer_backend,lora_qv_compatible,lora_target_counts,...,dtype_distribution_numel,bf16_parameter_ratio,dominant_dtype,attn_logit_softcapping,final_logit_softcapping,peak_allocated_gib,peak_reserved_gib,error_class,error_message,resume_from_checkpoint
0,M1,Llama 3.1 8B Instruct,meta-llama/Llama-3.1-8B-Instruct,0e9e39f249a16976918f6564b8830bc894c89659,True,4.039860,True,fast,True,"{""q_proj"": 32, ""v_proj"": 32}",...,"{""torch.bfloat16"": 8030261248}",1.0,torch.bfloat16,NaN,NaN,15.002958,15.156250,,,False
1,M2,Qwen2.5 7B Instruct,Qwen/Qwen2.5-7B-Instruct,a09a35458c702b33eeacc393d103063234e8bc28,True,3.759995,True,fast,True,"{""q_proj"": 28, ""v_proj"": 28}",...,"{""torch.bfloat16"": 7615616512}",1.0,torch.bfloat16,NaN,NaN,14.265306,14.388672,,,False
2,M3,Gemma 2 9B IT,google/gemma-2-9b-it,11c9b309abf73637e4b6f9a3fa1e92e615547819,True,4.786569,True,fast,True,"{""q_proj"": 42, ""v_proj"": 42}",...,"{""torch.bfloat16"": 9241705984}",1.0,torch.bfloat16,50.0,30.0,17.371922,17.400391,,,False
3,M4,Mistral 7B Instruct v0.3,mistralai/Mistral-7B-Instruct-v0.3,c170c708c41dac9275d15a8fff4eca08d52bab71,True,3.723584,True,fast,True,"{""q_proj"": 32, ""v_proj"": 32}",...,"{""torch.bfloat16"": 7248023552}",1.0,torch.bfloat16,NaN,NaN,13.546114,13.701172,,,False
4,M5,Aya Expanse 8B,CohereLabs/aya-expanse-8b,5062468bf9bc0c6035fd64e06274333ec127d980,True,6.538996,True,fast,True,"{""q_proj"": 32, ""v_proj"": 32}",...,"{""torch.bfloat16"": 8028033024}",1.0,torch.bfloat16,NaN,NaN,14.990824,15.125000,,,False



Benchmark loop selesai.
RUN_DIR: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_model_translator/runs/20260813T051727Z_88f26a8d


## 13. Menggabungkan hasil dan memeriksa kelengkapan


In [14]:
# ============================================================
# 13. GABUNGKAN HASIL DAN PERIKSA KELENGKAPAN
# ============================================================
frames = []
for key in MODELS:
    f = load_checkpoint(key)
    if not f.empty:
        frames.append(f)

if not frames:
    raise RuntimeError("Belum ada hasil benchmark pada RUN baru ini.")

df_predictions = pd.concat(frames, ignore_index=True)

for c in [
    "generation_success",
    "nonempty_output",
    "hit_max_new_tokens",
]:
    df_predictions[c] = (
        df_predictions[c]
        .astype(str)
        .str.lower()
        .eq("true")
    )

for c in [
    "prompt_tokens",
    "completion_tokens",
    "latency_seconds",
]:
    df_predictions[c] = pd.to_numeric(
        df_predictions[c], errors="coerce"
    )

df_predictions.to_csv(
    ALL_PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

df_completeness = (
    df_predictions
    .groupby(["model_key", "display_name"], as_index=False)
    .agg(
        n_rows=("row_id", "nunique"),
        generation_success_count=("generation_success", "sum"),
        nonempty_output_count=("nonempty_output", "sum"),
        hit_max_new_tokens_count=("hit_max_new_tokens", "sum"),
    )
)

display(df_completeness)

for key in MODELS:
    row = df_completeness[df_completeness["model_key"].eq(key)]
    if row.empty:
        print(f"{key}: tidak memiliki prediksi pada run ini.")
    else:
        r = row.iloc[0]
        print(
            f"{key}: {int(r['generation_success_count'])}/"
            f"{EXPECTED_VALID_ROWS} generasi berhasil."
        )


,model_key,display_name,n_rows,generation_success_count,nonempty_output_count,hit_max_new_tokens_count
0,M1,Llama 3.1 8B Instruct,53,53,53,0
1,M2,Qwen2.5 7B Instruct,53,53,53,0
2,M3,Gemma 2 9B IT,53,53,53,0
3,M4,Mistral 7B Instruct v0.3,53,53,53,2
4,M5,Aya Expanse 8B,53,53,53,0


M1: 53/53 generasi berhasil.
M2: 53/53 generasi berhasil.
M3: 53/53 generasi berhasil.
M4: 53/53 generasi berhasil.
M5: 53/53 generasi berhasil.


## 14. BLEU dan chrF++

chrF++ adalah metrik utama; BLEU digunakan sebagai pendukung/tie-breaker.


In [15]:
# ============================================================
# 14. BLEU, chrF++, DAN KELAYAKAN TEKNIS
# ============================================================
BLEU_METRIC = BLEU(tokenize="13a")
CHRFP_METRIC = CHRF(word_order=2)

def runtime_value(key: str, column: str, default=np.nan):
    rows = df_runtime[df_runtime["model_key"].eq(key)]
    return default if rows.empty else rows.iloc[-1].get(column, default)

def to_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() == "true"

summary = []

for key, info in MODELS.items():
    group = (
        df_predictions[df_predictions["model_key"].eq(key)]
        .copy()
        .sort_values("row_id", kind="stable")
    )

    n_rows = group["row_id"].nunique()
    success_count = int(group["generation_success"].sum()) if not group.empty else 0
    nonempty_count = int(group["nonempty_output"].sum()) if not group.empty else 0

    complete = (
        n_rows == EXPECTED_VALID_ROWS
        and success_count == EXPECTED_VALID_ROWS
        and nonempty_count == EXPECTED_VALID_ROWS
    )

    bleu = np.nan
    chrfpp = np.nan

    if complete:
        hypotheses = group["clean_output"].fillna("").astype(str).tolist()
        references = group["reference_text"].fillna("").astype(str).tolist()
        bleu = BLEU_METRIC.corpus_score(hypotheses, [references]).score
        chrfpp = CHRFP_METRIC.corpus_score(hypotheses, [references]).score

    load_success = to_bool(runtime_value(key, "load_success", False))
    lora_ok = to_bool(runtime_value(key, "lora_qv_compatible", False))
    smoke_ok = bool(SMOKE_SUCCESS.get(key, False))

    eligible = (
        smoke_ok
        and load_success
        and lora_ok
        and complete
    )

    summary.append({
        "model_key": key,
        "display_name": info["display_name"],
        "model_id": info["model_id"],
        "developer": info["developer"],
        "size": info["size"],
        "technical_eligible": eligible,
        "smoke_success": smoke_ok,
        "n_rows": n_rows,
        "generation_success_count": success_count,
        "nonempty_output_count": nonempty_count,
        "bleu": bleu,
        "chrfpp": chrfpp,
        "mean_latency_seconds": group["latency_seconds"].mean() if not group.empty else np.nan,
        "median_latency_seconds": group["latency_seconds"].median() if not group.empty else np.nan,
        "peak_allocated_gib": pd.to_numeric(
            pd.Series([runtime_value(key, "peak_allocated_gib", np.nan)]),
            errors="coerce",
        ).iloc[0],
        "peak_reserved_gib": pd.to_numeric(
            pd.Series([runtime_value(key, "peak_reserved_gib", np.nan)]),
            errors="coerce",
        ).iloc[0],
        "load_seconds": pd.to_numeric(
            pd.Series([runtime_value(key, "load_seconds", np.nan)]),
            errors="coerce",
        ).iloc[0],
        "lora_qv_compatible": lora_ok,
        "bf16_parameter_ratio": pd.to_numeric(
            pd.Series([runtime_value(key, "bf16_parameter_ratio", np.nan)]),
            errors="coerce",
        ).iloc[0],
        "dominant_dtype": runtime_value(key, "dominant_dtype", ""),
        "revision": MODEL_REVISIONS.get(key, "main"),
        "runtime_error": runtime_value(key, "error_message", ""),
    })

df_summary = pd.DataFrame(summary)

# Ranking metodologis: chrF++ utama, BLEU tie-breaker.
df_summary["chrfpp_rank"] = df_summary["chrfpp"].rank(
    ascending=False, method="min", na_option="bottom"
)
df_summary["bleu_rank"] = df_summary["bleu"].rank(
    ascending=False, method="min", na_option="bottom"
)

df_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(
    df_summary[[
        "model_key",
        "display_name",
        "technical_eligible",
        "bleu",
        "chrfpp",
        "chrfpp_rank",
        "bleu_rank",
        "mean_latency_seconds",
        "peak_allocated_gib",
        "lora_qv_compatible",
    ]]
    .sort_values(
        ["technical_eligible", "chrfpp", "bleu"],
        ascending=[False, False, False],
        kind="stable",
    )
    .round(4)
)

print("BLEU signature :", BLEU_METRIC.get_signature())
print("chrF++ signature:", CHRFP_METRIC.get_signature())


,model_key,display_name,technical_eligible,bleu,chrfpp,chrfpp_rank,bleu_rank,mean_latency_seconds,peak_allocated_gib,lora_qv_compatible
4,M5,Aya Expanse 8B,True,0.7264,20.9851,1.0,3.0,0.4209,14.9908,True
2,M3,Gemma 2 9B IT,True,3.2805,20.4617,2.0,1.0,0.8143,17.3719,True
3,M4,Mistral 7B Instruct v0.3,True,0.3642,19.5367,3.0,5.0,1.1030,13.5461,True
0,M1,Llama 3.1 8B Instruct,True,1.5350,18.5115,4.0,2.0,0.4539,15.0030,True
1,M2,Qwen2.5 7B Instruct,True,0.6001,18.3844,5.0,4.0,0.4751,14.2653,True


BLEU signature : nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.5.1
chrF++ signature: nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.5.1


## 15. Penetapan kandidat backbone dari hasil benchmark

Backbone **tidak dikunci sebelumnya**. Model yang dipilih harus:

1. lolos feasibility gate;
2. menyelesaikan 53/53 validation;
3. memiliki chrF++ tertinggi;
4. BLEU digunakan sebagai tie-breaker bila chrF++ sama.


In [16]:
# ============================================================
# 15. PILIH BACKBONE BERDASARKAN chrF++ → BLEU
# ============================================================
eligible = df_summary[df_summary["technical_eligible"]].copy()

if eligible.empty:
    raise RuntimeError(
        "Tidak ada model yang lolos feasibility gate lengkap. "
        "Jangan menetapkan backbone sebelum penyebab kegagalan diperiksa."
    )

eligible = eligible.sort_values(
    ["chrfpp", "bleu"],
    ascending=[False, False],
    kind="stable",
)

winner = eligible.iloc[0]

print("KANDIDAT BACKBONE TERPILIH DARI BENCHMARK VALIDATION")
print("---------------------------------------------------")
print("Kode                :", winner["model_key"])
print("Model               :", winner["display_name"])
print("chrF++ (utama)      :", round(float(winner["chrfpp"]), 4))
print("BLEU (pendukung)    :", round(float(winner["bleu"]), 4))
print("Technical eligible  :", bool(winner["technical_eligible"]))
print("LoRA q/v compatible :", bool(winner["lora_qv_compatible"]))

winner_record = {
    "run_id": RUN_ID,
    "selected_model_key": winner["model_key"],
    "selected_display_name": winner["display_name"],
    "selected_model_id": winner["model_id"],
    "selection_rule": "technical_eligible -> chrfpp_desc -> bleu_desc",
    "chrfpp": float(winner["chrfpp"]),
    "bleu": float(winner["bleu"]),
}

WINNER_PATH = RUN_DIR / "selected_backbone.json"
with open(WINNER_PATH, "w", encoding="utf-8") as f:
    json.dump(winner_record, f, ensure_ascii=False, indent=2)

print("\nDisimpan:", WINNER_PATH)


KANDIDAT BACKBONE TERPILIH DARI BENCHMARK VALIDATION
---------------------------------------------------
Kode                : M5
Model               : Aya Expanse 8B
chrF++ (utama)      : 20.9851
BLEU (pendukung)    : 0.7264
Technical eligible  : True
LoRA q/v compatible : True

Disimpan: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_model_translator/runs/20260813T051727Z_88f26a8d/selected_backbone.json


## 16. Ekspor laporan


In [17]:
# ============================================================
# 16. EKSPOR LAPORAN
# ============================================================
with pd.ExcelWriter(REPORT_XLSX_PATH, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_predictions.to_excel(writer, sheet_name="Predictions", index=False)
    df_runtime.to_excel(writer, sheet_name="Runtime", index=False)
    df_completeness.to_excel(writer, sheet_name="Completeness", index=False)
    if not df_preflight.empty:
        df_preflight.to_excel(writer, sheet_name="HF_Preflight", index=False)
    if not df_smoke.empty:
        df_smoke.to_excel(writer, sheet_name="Smoke_Test", index=False)

print("Benchmark selesai.")
print("RUN_ID :", RUN_ID)
print("Folder :", RUN_DIR)
print("Summary:", SUMMARY_PATH)
print("Excel  :", REPORT_XLSX_PATH)
print("Winner :", WINNER_PATH)


Benchmark selesai.
RUN_ID : 20260813T051727Z_88f26a8d
Folder : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_model_translator/runs/20260813T051727Z_88f26a8d
Summary: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_model_translator/runs/20260813T051727Z_88f26a8d/benchmark_summary.csv
Excel  : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_model_translator/runs/20260813T051727Z_88f26a8d/benchmark_model_report.xlsx
Winner : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/benchmark_model_translator/runs/20260813T051727Z_88f26a8d/selected_backbone.json


## 17. Tabel ringkas BAB 4


In [18]:
# ============================================================
# 17. TABEL RINGKAS BAB 4
# ============================================================
df_bab4 = (
    df_summary[[
        "model_key",
        "display_name",
        "generation_success_count",
        "nonempty_output_count",
        "bleu",
        "chrfpp",
        "mean_latency_seconds",
        "peak_allocated_gib",
        "bf16_parameter_ratio",
        "dominant_dtype",
        "lora_qv_compatible",
        "technical_eligible",
    ]]
    .copy()
    .rename(columns={
        "model_key": "Kode",
        "display_name": "Model",
        "generation_success_count": "Generasi Berhasil",
        "nonempty_output_count": "Keluaran Tidak Kosong",
        "bleu": "BLEU",
        "chrfpp": "chrF++",
        "mean_latency_seconds": "Rata-rata Inferensi (detik)",
        "peak_allocated_gib": "VRAM Puncak (GiB)",
        "bf16_parameter_ratio": "Rasio Parameter BF16",
        "dominant_dtype": "Dtype Dominan",
        "lora_qv_compatible": "Kompatibel LoRA q/v",
        "technical_eligible": "Layak Teknis",
    })
    .sort_values(
        ["Layak Teknis", "chrF++", "BLEU"],
        ascending=[False, False, False],
        kind="stable",
    )
)

display(df_bab4.round(4))


,Kode,Model,Generasi Berhasil,Keluaran Tidak Kosong,BLEU,chrF++,Rata-rata Inferensi (detik),VRAM Puncak (GiB),Rasio Parameter BF16,Dtype Dominan,Kompatibel LoRA q/v,Layak Teknis
4,M5,Aya Expanse 8B,53,53,0.7264,20.9851,0.4209,14.9908,1.0,torch.bfloat16,True,True
2,M3,Gemma 2 9B IT,53,53,3.2805,20.4617,0.8143,17.3719,1.0,torch.bfloat16,True,True
3,M4,Mistral 7B Instruct v0.3,53,53,0.3642,19.5367,1.1030,13.5461,1.0,torch.bfloat16,True,True
0,M1,Llama 3.1 8B Instruct,53,53,1.5350,18.5115,0.4539,15.0030,1.0,torch.bfloat16,True,True
1,M2,Qwen2.5 7B Instruct,53,53,0.6001,18.3844,0.4751,14.2653,1.0,torch.bfloat16,True,True
